# 🏠 House Card Asset Generator

Generates a consistent set of isometric house images for the
[House Card](https://github.com/muhlman/house_card) (a fork of
[silasmariusz/fork_u-house_card](https://github.com/silasmariusz/fork_u-house_card)).

**What it makes**
- Base images for all 4 seasons, day and night (8).
- Weather variants: overcast, rain, snow, fog, lightning, hail - day and night, every season (48, or 44 with the unlikely combinations skipped).
- Two Christmas images (14 Dec - 14 Jan).
- Optional: up to two **vehicle overlays** (transparent PNGs placed on the driveway while a person is home), with day/night and weather variants. Each is added to your finished house image as a local edit and cut out by difference, so it lands in the right place at the right scale.
- Optional: **event images** (4th of July, birthday, Halloween, Thanksgiving, New Year, Easter, ...) that the card swaps in via its `events:` config.
- Optional: the upstream "gaming mode" images (the card does not use them).

Every output is resized to the master's pixel size so the card and the overlays line up exactly.

### Instructions
1. **API key**: create one at <https://aistudio.google.com/apikey> and paste it in step 1. The free tier is enough.
2. **Reference photos**: step 2 asks for 1-5 photos of your house (street view, side view, satellite / roof).
3. **Master**: step 4 renders the summer-day master. Re-run it until you like it - every other image copies its geometry.
4. **Run the rest**: steps 5-7b generate, step 8 shows thumbnails, step 9 downloads a zip. Already-generated files are skipped, so you can re-run after a rate limit or delete a file to redo it.
5. Copy the PNGs to the folder your card's `image_path` points at (default `config/www/community/house_card/images/`).


In [ ]:
#@title 1. Setup & Configuration
#@markdown Paste your API key (https://aistudio.google.com/apikey) and describe your house.

import io, os, re, time, zipfile
from pathlib import Path
from google.colab import files

!pip install -q -U google-genai
from google import genai
from google.genai import types
from PIL import Image
import logging, warnings
logging.getLogger("google_genai.models").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=DeprecationWarning)

GOOGLE_API_KEY = "" #@param {type:"string"}
HOUSE_DESCRIPTION = "White two-story colonial with black shutters, red brick base, attached two-car garage on the right, paver driveway" #@param {type:"string"}

#@markdown **Model.** `gemini-nano-banana-2.1` is the current fast model (1K/2K/4K); `gemini-3-pro-image` is the higher-fidelity Nano Banana Pro; `gemini-2.5-flash-image` is the older free-tier model.
MODEL_TYPE = "gemini-nano-banana-2.1" #@param ["gemini-nano-banana-2.1", "gemini-3-pro-image", "gemini-3.1-flash-image", "gemini-2.5-flash-image"]
#@markdown **Frame.** Keep 4:3 unless you also change `aspect_ratio` in the card config. 1K is plenty for a dashboard.
ASPECT_RATIO = "4:3" #@param ["4:3", "3:2", "16:9", "1:1"]
IMAGE_SIZE = "1K" #@param ["1K", "2K", "4K"]

#@markdown **What to generate.**
GENERATE_RARE_WEATHER = False #@param {type:"boolean"}
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;Also make summer snow and winter hail. Off by default: the card falls back to the base image when a file is missing.
GENERATE_XMAS = True #@param {type:"boolean"}
GENERATE_EVENTS = False #@param {type:"boolean"}
EVENTS_TO_GENERATE = "july4, birthday, halloween, thanksgiving, new_year" #@param {type:"string"}
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;Event images swap the whole house via the card's `events:` config. Available: july4, birthday, halloween, thanksgiving, new_year, easter, valentines, st_patricks, graduation, game_day. Birthday is made for all four seasons.
GENERATE_GAMING = False #@param {type:"boolean"}
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;Upstream "gaming mode" images; the card does not display them.

#@markdown **Vehicle overlays.** Leave a description empty to skip that vehicle. The house images are then
#@markdown generated with an **empty driveway** so the overlays always land on bare pavement.
VEHICLE_1_DESCRIPTION = "black Ford F-150 pickup truck, gloss black" #@param {type:"string"}
VEHICLE_1_PARKING_SPOT = "on the paver driveway in front of the LEFT garage door, nose towards the garage" #@param {type:"string"}
VEHICLE_2_DESCRIPTION = "white Hyundai Palisade SUV, pearl white" #@param {type:"string"}
VEHICLE_2_PARKING_SPOT = "on the paver driveway in front of the RIGHT garage door, nose towards the garage" #@param {type:"string"}
VEHICLE_WEATHER_VARIANTS = True #@param {type:"boolean"}
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;Off = only day and night per vehicle (2 files). On = also overcast, rainy, snowy, fog, lightning, hail (12 files).
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;Give each vehicle its own spot so both can show at once without overlapping. Describe the spot relative to
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;fixed features of the master image (garage doors, front walk, mailbox).
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;Optional regions (`left,top,right,bottom` in % of the frame) limit where each vehicle may be extracted from.
#@markdown &nbsp;&nbsp;&nbsp;&nbsp;Leave empty to keep the largest changed area; step 8 prints the region each vehicle was found in.
VEHICLE_1_REGION = "" #@param {type:"string"}
VEHICLE_2_REGION = "" #@param {type:"string"}

def _region(txt):
    nums = [float(x) for x in re.findall(r"[\d.]+", txt or "")]
    return tuple(nums) if len(nums) == 4 else None
VEHICLES = [(d.strip(), s.strip(), _region(r)) for d, s, r in
            ((VEHICLE_1_DESCRIPTION, VEHICLE_1_PARKING_SPOT, VEHICLE_1_REGION),
             (VEHICLE_2_DESCRIPTION, VEHICLE_2_PARKING_SPOT, VEHICLE_2_REGION)) if d.strip()]
EMPTY_DRIVEWAY = len(VEHICLES) > 0

if not GOOGLE_API_KEY:
    raise SystemExit("❌ Please enter your API key in the form above and run this cell again.")
client = genai.Client(api_key=GOOGLE_API_KEY)
EVENTS_SELECTED = [e.strip().lower() for e in EVENTS_TO_GENERATE.split(",") if e.strip()] if GENERATE_EVENTS else []
print(f"✅ Model {MODEL_TYPE}, {ASPECT_RATIO} at {IMAGE_SIZE}; vehicles: {len(VEHICLES)}; empty driveway: {EMPTY_DRIVEWAY}; events: {EVENTS_SELECTED or 'none'}")

BASE_DIR = Path("/content/house_assets")
REF_DIR, OUT_DIR, MASTER_DIR, VEHICLE_DIR = (BASE_DIR / d for d in ("reference", "output", "master", "output/overlays"))
for d in (REF_DIR, OUT_DIR, MASTER_DIR, VEHICLE_DIR):
    d.mkdir(parents=True, exist_ok=True)
MASTER_PATH = MASTER_DIR / "_master_reference.png"


In [ ]:
#@title 2. Upload Reference Images
#@markdown Upload 1-5 photos of your house. Ideally: 1 front view, 1 side view, 1 isometric/satellite view if available.

uploaded = files.upload()

for filename in uploaded.keys():
    # Move to reference dir
    os.rename(filename, REF_DIR / filename)
    print(f"Saved reference: {filename}")

print(f"✅ Total references: {len(list(REF_DIR.glob('*')))}")

In [ ]:
#@title 3. Prompts & generation helpers

SEASONS = ["winter", "spring", "summer", "autumn"]
TIMES = ["day", "night"]
# Suffixes exactly as the card requests them ({season}_{weather}_{time}.png)
WEATHERS = ["overcast", "rainy", "snowy", "fog", "lightning", "hail"]
UNLIKELY = {("summer", "snowy"), ("winter", "hail")}

WEATHER_RULES = {
    "sunny":     "Clear sky: plot highly illuminated by golden sun rays (day) or a crisp clear starry night.",
    "overcast":  "Overcast: flat grey cloud cover, soft diffuse light, no sun shafts, muted colours. No rain.",
    "rainy":     "Heavy rain: a downpour over the plot, wet reflective surfaces, puddles with light reflections.",
    "snowy":     "Snowing: intense snowfall over the plot, fresh snow accumulating on roof, ground and surfaces.",
    "hail":      "Hailstorm: large chunks of ice bouncing and lying on the ground and roof instead of snow, random puddles under the ice.",
    "lightning": "Thunderstorm: dark storm sky, the house strongly overexposed by lightning flashes, hard low-angle shadows. Do not draw lightning bolts.",
    "fog":       "Fog: a delicate cloud layer and slight haze in front of the house and across the plot, softened contrast.",
}
SEASON_GROUND = {
    "winter": "Winter: ground and roof covered in snow, bare trees. No grass.",
    "spring": "Spring: fresh green grass with a few spring flowers, trees in light leaf.",
    "summer": "Summer: manicured vibrant green lawn, full foliage.",
    "autumn": "Autumn: a moderate scatter of yellow-orange-brown leaves on the grass, trees in autumn colour.",
}
GAMING_MODES = {
    "synthwave": "Synthwave style: neon purples/pinks, grid lines, retro-future vibe. Flying DeLorean parked in front.",
    "cyberpunk": "Cyberpunk style: high-tech low-life, neon signs, rain, dark gritty atmosphere. Flying DeLorean parked in front.",
    "matrix":    "Matrix style: green code rain, digital artifacts. Half the asset in green wireframe vector style.",
    "mario":     "Mario Bros World style: green pipe, gold coins/stars on roof, beanstalk to sky. Mario & Luigi grilling in front of the house.",
    "xbox_kid":  "Small astronaut kid (big head) playing Xbox on a red semi-transparent sofa in front of the house. Projector screen.",
}
# Event images: (season the scene is set in, or None = one per season; prompt extras)
EVENTS = {
    "july4":        ("summer", "4th of July: American flags and red-white-blue bunting on the porch and railings, a flag on a pole, "
                               "fireworks bursting in the sky above the house, a few lawn chairs and a small grill on the patio, sparkler light trails."),
    "birthday":     (None,     "Birthday party: colourful balloons tied to the porch, mailbox and railings, a 'HAPPY BIRTHDAY' banner across the front, "
                               "streamers, a table with a birthday cake and presents on the patio, confetti on the lawn."),
    "halloween":    ("autumn", "Halloween: carved glowing jack-o'-lanterns on the steps, cobwebs on the porch, a skeleton by the door, "
                               "orange and purple string lights, a few gravestone props on the lawn, bats in the sky."),
    "thanksgiving": ("autumn", "Thanksgiving: harvest decorations - pumpkins, gourds and corn stalks by the door, a wreath of autumn leaves, "
                               "warm light in every window, a long table set for dinner visible through the windows."),
    "new_year":     ("winter", "New Year's Eve: fireworks and confetti over the house, a glowing '2026' style numerals sign on the lawn, champagne-gold "
                               "string lights, party hats and streamers on the porch, snow on the ground."),
    "easter":       ("spring", "Easter: pastel-coloured eggs hidden across the lawn and flower beds, a large bunny figure by the porch, "
                               "pastel bunting, spring flowers in bloom, a basket of eggs on the steps."),
    "valentines":   ("winter", "Valentine's Day: red and pink heart decorations and a heart wreath on the door, pink string lights along the eaves, "
                               "rose petals on the path, warm romantic lighting."),
    "st_patricks":  ("spring", "St. Patrick's Day: green bunting and shamrock decorations, a rainbow ending in a pot of gold on the lawn, "
                               "green string lights, a leprechaun hat on the mailbox."),
    "graduation":   ("spring", "Graduation party: a 'CONGRATS GRAD' banner across the porch, balloons in school colours, a mortarboard and "
                               "diploma on the lawn sign, a table with cake and a photo board on the patio."),
    "game_day":     ("autumn", "Game day: team pennants and a 'GAME DAY' banner, a tailgate setup on the driveway edge with a grill, "
                               "cooler and folding chairs, a projector screen on the garage showing a football field, foam fingers."),
}

XMAS_EXTRAS = {
    "day":   "Christmas: Santa sliding down the roof, penguins, an igloo in the garden, festive decorations and wreaths.",
    "night": "Christmas: Santa sliding down the roof, penguins, an igloo in the garden, strings of glowing Christmas lights on the eaves.",
}

DRIVEWAY_RULE = ("- DRIVEWAY: completely EMPTY. No vehicles, bins, planters or objects on the driveway or in front of the garage. "
                 "Bare pavement only." if EMPTY_DRIVEWAY else "")

BASE_RULES = f"""
RULES (CAMERA - MANDATORY):
- VIEW: STRICT ISOMETRIC ONLY (SimCity / The Sims style). Equal angles, parallel lines stay parallel.
- The house and plot are shown in this fixed isometric projection. NEVER render from a different angle.
- MARGINS: 15% left, 15% right, 5% top, 15% bottom. The asset is centred within these margins with the whole glassmorphism base visible. Weather effects may extend beyond the margins.
- ASPECT RATIO: {ASPECT_RATIO}.
{DRIVEWAY_RULE}

RULES (VISUAL STYLE):
- Ultra-high-fidelity 3D asset, "SimCity 2030" / "The Sims 5" aesthetic. Dramatic, cinematic, high-contrast lighting.
- Volumetric light only on the asset, never on the background. Soft bloom, ray-traced reflections on glass, wet surfaces and paint.
- A thick multi-layered glassmorphism base plate at the very bottom, with soil/grass/snow on top.
- Background: solid #121212 everywhere outside the plot, for a clean cut-out. Nothing drawn on the background.
- Quality: maximum sharpness, zero blur, no text, no watermark.

RULES (FIDELITY):
- House: {HOUSE_DESCRIPTION}
- CONSISTENCY: geometry, camera angle, position and scale 100% identical to the master reference. Use references only for geometry and architectural detail; re-draw the scene, do not filter the reference.
"""

def master_prompt():
    return f"""Task: create a "SimCity 2030" style 3D isometric asset of this house from the attached photos.
{BASE_RULES}
This image: Summer, Day, {WEATHER_RULES['sunny']} {SEASON_GROUND['summer']}
"""

def event_prompt(season, time_of_day, extras):
    return variant_prompt(season, time_of_day, "sunny", "Event decorations (keep the house itself unchanged): " + extras)

def variant_prompt(season, time_of_day, weather="sunny", extras=""):
    night = "Night: deep navy / blue-grey grading on the whole plot, warm lights on inside the house, driveway and path lamps lit." if time_of_day == "night" else "Day."
    return f"""Reference: the attached master image. Same isometric angle, position and scale - redraw it for the conditions below.
{BASE_RULES}
This image:
- Season: {season.capitalize()}. {SEASON_GROUND[season]}
- Time: {night}
- Weather: {WEATHER_RULES[weather]}
{extras}
"""

def vehicle_prompt(desc, spot, time_of_day, weather):
    """Local edit of a finished house image: the model keeps the scene and adds the vehicle."""
    wx = "" if weather == "sunny" else (f"Weather effect on the vehicle only: {WEATHER_RULES[weather]} "
                                         "Show it on the vehicle (wet paint, snow or ice on roof and hood, haze, flash highlights).")
    light = ("Night: the vehicle is lit by the driveway lamps and house lights with the same blue-grey ambient as the scene."
             if time_of_day == "night" else "Match the scene's daylight and shadow direction exactly.")
    return f"""Edit the attached image. Add ONE {desc}, parked {spot}, drawn at the exact scale and isometric angle of the scene, with a soft contact shadow under it.
Change NOTHING else: the house, roof, trees, lawn, driveway, lighting, sky and background must stay pixel-identical to the attached image. Do not move the camera, do not re-render the scene, do not add other objects.
{light}
{wx}
Same style as the scene (ultra-high-fidelity 3D asset, gloss paint with ray-traced reflections). No text, no watermark.
"""

def _config():
    """image_config is the documented way to pin aspect ratio / size; fall back if the SDK or model rejects it."""
    no_afc = dict(automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
    attempts = [
        dict(response_modalities=["IMAGE"], image_config=types.ImageConfig(aspect_ratio=ASPECT_RATIO, image_size=IMAGE_SIZE)),
        dict(response_modalities=["IMAGE"], image_config=types.ImageConfig(aspect_ratio=ASPECT_RATIO)),
        dict(response_modalities=["IMAGE"]),
    ]
    for kw in attempts:
        try:
            yield types.GenerateContentConfig(**kw, **no_afc)
        except (TypeError, ValueError):
            continue

def _extract_image(response):
    for part in getattr(response, "parts", None) or (response.candidates[0].content.parts if response.candidates else []):
        if getattr(part, "inline_data", None) and part.inline_data.data:
            return Image.open(io.BytesIO(part.inline_data.data))
    return None

def generate_image(prompt, ref_paths, out_path, match_master=True, retries=3):
    """Generate one image, resize it to the master's size (so every file lines up), save as PNG."""
    contents = [prompt] + [Image.open(p) for p in ref_paths if Path(p).exists()]
    for attempt in range(retries):
        for cfg in _config():
            try:
                response = client.models.generate_content(model=MODEL_TYPE, contents=contents, config=cfg)
            except Exception as e:  # noqa: BLE001
                msg = str(e)
                if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
                    print("   rate limit - waiting 60 s"); time.sleep(60); break
                if "400" in msg and ("image_config" in msg or "image_size" in msg or "aspect" in msg):
                    continue  # try the next, simpler config
                print(f"   error: {msg[:200]}"); break
            img = _extract_image(response)
            if img is None:
                print("   no image in response"); break
            if match_master and MASTER_PATH.exists() and out_path != MASTER_PATH:
                size = Image.open(MASTER_PATH).size
                if img.size != size:
                    img = img.resize(size, Image.LANCZOS)
            img.convert("RGB").save(out_path, "PNG")
            return True
    return False

def extract_vehicle(edited_path, base_path, dst_path, region=None, threshold=28, min_area_pct=0.15):
    """Alpha-mask the pixels that changed between the base scene and the edited scene.
    Keeps the largest changed blob (or the blobs inside `region`, given in % of the frame) so
    small re-render noise elsewhere is dropped. Writes a full-frame RGBA PNG. Returns (coverage %, bbox %)."""
    import numpy as np
    from PIL import ImageFilter
    base = Image.open(base_path).convert("RGB")
    edit = Image.open(edited_path).convert("RGB")
    if edit.size != base.size:
        edit = edit.resize(base.size, Image.LANCZOS)
    a = np.asarray(base).astype(np.int16); b = np.asarray(edit).astype(np.int16)
    diff = np.abs(a - b).max(axis=2)
    mask = diff > threshold
    # close small gaps / drop specks
    m = Image.fromarray((mask * 255).astype(np.uint8)).filter(ImageFilter.MaxFilter(7)).filter(ImageFilter.MinFilter(7))
    mask = np.asarray(m) > 127
    H, W = mask.shape
    try:
        from scipy import ndimage
        labels, n = ndimage.label(mask)
        if n:
            sizes = ndimage.sum(mask, labels, range(1, n + 1))
            keep = np.zeros(n + 1, bool)
            if region:
                l, t, r, btm = (region[0] * W / 100, region[1] * H / 100, region[2] * W / 100, region[3] * H / 100)
                for i, sl in enumerate(ndimage.find_objects(labels), 1):
                    ys, xs = sl
                    if xs.stop > l and xs.start < r and ys.stop > t and ys.start < btm and sizes[i - 1] > min_area_pct / 100 * H * W:
                        keep[i] = True
            if not keep.any():
                keep[int(np.argmax(sizes)) + 1] = True
            mask = keep[labels]
    except ImportError:
        if region:
            l, t, r, btm = (int(region[0] * W / 100), int(region[1] * H / 100), int(region[2] * W / 100), int(region[3] * H / 100))
            box = np.zeros_like(mask); box[t:btm, l:r] = True; mask &= box
    alpha = Image.fromarray((mask * 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(1.2))
    out = Image.new("RGBA", base.size)
    out.paste(edit, (0, 0)); out.putalpha(alpha)
    out.save(dst_path, "PNG")
    ys, xs = np.where(mask)
    bbox = (round(xs.min() / W * 100), round(ys.min() / H * 100), round(xs.max() / W * 100), round(ys.max() / H * 100)) if len(xs) else None
    return float(mask.mean()) * 100, bbox

print("✅ Prompts and helpers ready.")


In [ ]:
#@title 4. Generate the master reference (Summer, Day)
#@markdown Everything else copies this image's geometry, so re-run until you are happy with it.
refs = sorted(p for p in REF_DIR.glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp"))
if not refs:
    print("⚠️ No reference photos uploaded - the model will invent a house from the description.")
REGENERATE_MASTER = False #@param {type:"boolean"}
if MASTER_PATH.exists() and not REGENERATE_MASTER:
    print("Master already exists - tick REGENERATE_MASTER to redo it.")
elif generate_image(master_prompt(), refs, MASTER_PATH, match_master=False):
    print(f"✅ Master generated: {Image.open(MASTER_PATH).size[0]}x{Image.open(MASTER_PATH).size[1]}")
else:
    print("❌ Master generation failed - check the key / model and run again.")
if MASTER_PATH.exists():
    from IPython.display import display
    display(Image.open(MASTER_PATH).resize((600, int(600 * Image.open(MASTER_PATH).size[1] / Image.open(MASTER_PATH).size[0]))))


In [ ]:
#@title 5. Generate the house set (base, weather, Christmas)
if not MASTER_PATH.exists():
    raise SystemExit("❌ No master image. Run step 4 first.")

plan = []  # (filename, prompt)
for season in SEASONS:
    for t in TIMES:
        plan.append((f"{season}_{t}.png", variant_prompt(season, t)))
        for w in WEATHERS:
            if not GENERATE_RARE_WEATHER and (season, w) in UNLIKELY:
                continue
            plan.append((f"{season}_{w}_{t}.png", variant_prompt(season, t, w)))
if GENERATE_XMAS:
    for t in TIMES:
        plan.append((f"winter_xmas_{t}.png", variant_prompt("winter", t, "snowy", XMAS_EXTRAS[t])))

todo = [(f, p) for f, p in plan if not (OUT_DIR / f).exists()]
print(f"{len(plan)} house images in the plan, {len(plan) - len(todo)} already done, {len(todo)} to generate.")
failed = []
for i, (fname, prompt) in enumerate(todo, 1):
    print(f"[{i}/{len(todo)}] {fname}")
    if not generate_image(prompt, [MASTER_PATH], OUT_DIR / fname):
        failed.append(fname)

skipped = [f"{s}_{w}_{t}.png" for s in SEASONS for t in TIMES for w in WEATHERS if not GENERATE_RARE_WEATHER and (s, w) in UNLIKELY]
print(f"\n✅ Done. Failed: {failed or 'none'}")
if skipped:
    print(f"Skipped as rare weather (card falls back to the base image; tick GENERATE_RARE_WEATHER to make them): {', '.join(skipped)}")


In [ ]:
#@title 6. Generate vehicle overlays (optional)
#@markdown Each vehicle is added to the finished summer day / night house as a **local edit**, then the pixels that
#@markdown changed are cut out into a full-frame transparent PNG. Because the edit starts from your own house
#@markdown image, the vehicle lands at the right scale and in the right place. Check step 8: a coverage well
#@markdown above ~10% means the model re-rendered more than the vehicle - delete that file and re-run, or set a
#@markdown `VEHICLE_N_REGION` so only changes in that part of the frame are kept.
if not VEHICLES:
    print("No vehicle descriptions set - skipping.")
elif not MASTER_PATH.exists():
    raise SystemExit("❌ No master image. Run step 4 first.")
else:
    RAW_DIR = VEHICLE_DIR / "_raw"; RAW_DIR.mkdir(exist_ok=True)
    weathers = ["sunny"] + (WEATHERS if VEHICLE_WEATHER_VARIANTS else [])
    vplan = []
    for n, (desc, spot, region) in enumerate(VEHICLES, 1):
        for t in TIMES:
            base = OUT_DIR / f"summer_{t}.png"
            if not base.exists():
                print(f"⚠️ {base.name} missing - run step 5 first; skipping vehicle {n} {t}"); continue
            for w in weathers:
                fname = f"vehicle_{n}_{t}.png" if w == "sunny" else f"vehicle_{n}_{w}_{t}.png"
                vplan.append((fname, desc, spot, region, t, w, base))
    todo = [v for v in vplan if not (VEHICLE_DIR / v[0]).exists()]
    print(f"{len(vplan)} overlay images planned, {len(vplan) - len(todo)} done, {len(todo)} to generate.")
    for i, (fname, desc, spot, region, t, w, base) in enumerate(todo, 1):
        print(f"[{i}/{len(todo)}] {fname}")
        raw = RAW_DIR / fname
        if generate_image(vehicle_prompt(desc, spot, t, w), [base], raw):
            cov, bbox = extract_vehicle(raw, base, VEHICLE_DIR / fname, region)
            flag = "  ⚠️ large changed area - the model re-rendered more than the vehicle" if cov > 10 else ("  ⚠️ almost nothing changed - no vehicle found" if cov < 0.2 else "")
            print(f"   extracted {cov:.1f}% of the frame, region {bbox}{flag}")
    print("\n✅ Overlays done. Files are in output/overlays/.")


In [ ]:
#@title 7. Generate gaming-mode images (optional, unused by the card)
if not GENERATE_GAMING:
    print("GENERATE_GAMING is off - skipping.")
else:
    for mode, desc in GAMING_MODES.items():
        fname = f"gaming_{mode}.png"
        if not (OUT_DIR / fname).exists():
            print(fname)
            generate_image(desc + "\n" + BASE_RULES, [MASTER_PATH], OUT_DIR / fname)
    print("✅ Gaming images done.")


In [ ]:
#@title 7b. Generate event images (optional)
#@markdown Full house renders for holidays and parties, shown by the card's `events:` config.
#@markdown Files: `event_{name}_{day|night}.png`; birthday is per season: `event_birthday_{season}_{day|night}.png`.
if not EVENTS_SELECTED:
    print("GENERATE_EVENTS is off or no events listed - skipping.")
elif not MASTER_PATH.exists():
    raise SystemExit("❌ No master image. Run step 4 first.")
else:
    unknown = [e for e in EVENTS_SELECTED if e not in EVENTS]
    if unknown:
        print(f"⚠️ Unknown events ignored: {unknown}. Available: {', '.join(EVENTS)}")
    eplan = []
    for name in EVENTS_SELECTED:
        if name not in EVENTS: continue
        season, extras = EVENTS[name]
        seasons = SEASONS if season is None else [season]
        for s in seasons:
            for t in TIMES:
                fname = f"event_{name}_{t}.png" if season else f"event_{name}_{s}_{t}.png"
                eplan.append((fname, event_prompt(s, t, extras)))
    todo = [(f, p) for f, p in eplan if not (OUT_DIR / f).exists()]
    print(f"{len(eplan)} event images planned, {len(eplan) - len(todo)} done, {len(todo)} to generate.")
    for i, (fname, prompt) in enumerate(todo, 1):
        print(f"[{i}/{len(todo)}] {fname}")
        generate_image(prompt, [MASTER_PATH], OUT_DIR / fname)
    print("\n✅ Event images done.")


In [ ]:
#@title 8. Review thumbnails
#@markdown Overlays are shown composited over the matching house image so you can check alignment.
from IPython.display import display

def sheet(paths, cols=4, w=320, bg=None):
    paths = [p for p in paths if p.exists()]
    if not paths: return None
    first = Image.open(paths[0]); h = int(w * first.size[1] / first.size[0])
    rows = (len(paths) + cols - 1) // cols
    canvas = Image.new("RGB", (cols * w, rows * (h + 18)), (18, 18, 18))
    try:
        from PIL import ImageDraw
        draw = ImageDraw.Draw(canvas)
    except Exception:  # noqa: BLE001 - a half-upgraded Pillow can break ImageDraw; labels are optional
        draw = None
    for i, p in enumerate(paths):
        img = Image.open(p).convert("RGBA")
        if bg is not None:
            base = Image.open(bg(p.name)).convert("RGBA") if Path(bg(p.name)).exists() else Image.new("RGBA", img.size, (40, 40, 40, 255))
            img = Image.alpha_composite(base, img)
        img = img.convert("RGB").resize((w, h))
        x, y = (i % cols) * w, (i // cols) * (h + 18)
        canvas.paste(img, (x, y + 18))
        if draw: draw.text((x + 4, y + 2), p.name, fill=(230, 230, 230))
        else: print(f"  [{i}] {p.name}")
    return canvas

houses = sorted(OUT_DIR.glob("*.png"))
print(f"{len(houses)} house images"); s = sheet(houses); display(s) if s else None

def house_for(overlay_name):
    m = re.match(r"vehicle_\d+(?:_(\w+?))?_(day|night)\.png", overlay_name)
    w, t = (m.group(1), m.group(2)) if m else (None, "day")
    cand = OUT_DIR / (f"summer_{w}_{t}.png" if w else f"summer_{t}.png")
    return cand if cand.exists() else OUT_DIR / f"summer_{t}.png"

overlays = sorted(VEHICLE_DIR.glob("vehicle_*.png"))
print(f"{len(overlays)} overlay images"); s = sheet(overlays, bg=house_for); display(s) if s else None
import numpy as np
for p in overlays:
    a = np.asarray(Image.open(p).convert("RGBA"))[..., 3] > 127
    ys, xs = np.where(a); H, W = a.shape
    if len(xs):
        print(f"  {p.name}: {a.mean()*100:.1f}% opaque, region {round(xs.min()/W*100)},{round(ys.min()/H*100)},{round(xs.max()/W*100)},{round(ys.max()/H*100)}")

# Both vehicles at once (the "everyone home" case) - check they sit in separate spots
for t in TIMES:
    parts = [VEHICLE_DIR / f"vehicle_{n}_{t}.png" for n in range(1, len(VEHICLES) + 1)]
    base = OUT_DIR / f"summer_{t}.png"
    if len(parts) > 1 and all(p.exists() for p in parts) and base.exists():
        img = Image.open(base).convert("RGBA")
        for p in parts:
            img = Image.alpha_composite(img, Image.open(p).convert("RGBA"))
        print(f"All vehicles, {t}:"); display(img.convert("RGB").resize((640, int(640 * img.size[1] / img.size[0]))))


In [ ]:
#@title 9. Download everything as a zip
#@markdown Copy the contents of `house_card_images.zip` into the card's image folder (default `config/www/community/house_card/images/`).
zip_path = BASE_DIR / "house_card_images.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in OUT_DIR.glob("*.png"): zf.write(f, f.name)
    for f in VEHICLE_DIR.glob("vehicle_*.png"): zf.write(f, f.name)
    if MASTER_PATH.exists(): zf.write(MASTER_PATH, "_master_reference.png")
n = len(list(OUT_DIR.glob("*.png"))) + len(list(VEHICLE_DIR.glob("vehicle_*.png")))
print(f"Zipped {n} images.")
files.download(str(zip_path))
